# DynaICL Stage 0.5: confirmatory population-advantage run (Google Colab)

Runs the **pre-registered** confirmatory experiment (`docs/PREREG_stage05.md`, frozen, sha256 `fcf7f3a55c5a58de...`):
matched population-aware Bayes vs the strongest context-only frontier (primary), and finite library vs
continuous prior (secondary). 144 cells x 2 independent replicates x 3378 items. CPU only.

**Before you start**
* `Runtime -> Change runtime type -> CPU`. **No GPU is needed.**
* Expected time on the free tier (2 vCPU): about **2-2.5 h**. This is estimated from the local pilot, not measured on Colab.
* **Interruption-safe:** each finished cell is saved to Google Drive (`MyDrive/dynaicl_stage05/checkpoints`).
  If the session dies, open the notebook again, do `Runtime -> Run all`, upload the zip again, and it **resumes**.
  The results are identical to an uninterrupted run. The final results are written straight to Drive.

**You need:** `dynaicl_stage0_bundle.zip` (sha256 `5ace85ebf8bc81b6...`, 83 files). Then `Runtime -> Run all`.

In [ ]:
!python3 --version
!nproc

## 1. Mount Drive (needed for checkpoints and results) and get the code bundle

In [ ]:
import hashlib, os, zipfile
from google.colab import drive
drive.mount("/content/drive")
os.makedirs("/content/drive/MyDrive/dynaicl_stage05", exist_ok=True)

BUNDLE_ON_DRIVE = ""          # optional: e.g. "/content/drive/MyDrive/dynaicl_stage0_bundle.zip"
EXPECTED = {"bundle": "5ace85ebf8bc81b617279312994b69eb130ffb814fb3ca37c8d9d8c28503e117", "prereg": "fcf7f3a55c5a58de6ba3c4ea75c30f5ed971fccaee4fa6b2b8db852d30751cbf", "config_hash": "51c7621b7a8ff46cb81e70be7a1fa3b9213846858742f6b3b19ad4f6b9792ef3"}
if BUNDLE_ON_DRIVE:
    src = BUNDLE_ON_DRIVE
else:
    from google.colab import files
    src = next(iter(files.upload()))       # pick dynaicl_stage0_bundle.zip
sha = hashlib.sha256(open(src, "rb").read()).hexdigest()
print("bundle sha256 matches:", sha == EXPECTED["bundle"])
os.makedirs("/content/DynaICL", exist_ok=True)
zipfile.ZipFile(src).extractall("/content/DynaICL")
os.chdir("/content/DynaICL")
bad = [rel for h, rel in (l.rstrip(chr(10)).split("  ", 1) for l in open("MANIFEST.sha256"))
       if hashlib.sha256(open(rel, "rb").read()).hexdigest() != h]
print("manifest check:", "OK" if not bad else f"MISMATCH {bad}")
print("pre-registration sha256 matches the frozen one:",
      hashlib.sha256(open("docs/PREREG_stage05.md", "rb").read()).hexdigest() == EXPECTED["prereg"])

## 2. Install the pinned stack (pip conflict warnings about other Colab packages are harmless here)

In [ ]:
!pip install -q -r /content/DynaICL/requirements.txt
!python3 -c "import numpy, scipy, yaml, pytest; print('numpy', numpy.__version__, '| scipy', scipy.__version__, '| pyyaml', yaml.__version__, '| pytest', pytest.__version__)"

## 3. Quick checks (about 1 min): Stage 0.5 tests and the config hash

In [ ]:
%cd /content/DynaICL
!python3 -m pytest -q -p no:cacheprovider tests/test_stage05.py
!python3 -c "from dynaicl.config.schema import load_population_config as L; h=L('config/stage05.yaml').config_hash(); print('config hash', h[:12], 'OK' if h=='51c7621b7a8ff46cb81e70be7a1fa3b9213846858742f6b3b19ad4f6b9792ef3' else 'MISMATCH')"

## 4. The confirmatory run (the long step; prints one line per finished cell; resumes from Drive checkpoints)

In [ ]:
%cd /content/DynaICL
!OPENBLAS_NUM_THREADS=1 OMP_NUM_THREADS=1 MKL_NUM_THREADS=1 python3 -m dynaicl.experiments.population_advantage --config config/stage05.yaml --workers $(nproc) --checkpoint-dir /content/drive/MyDrive/dynaicl_stage05/checkpoints --runs-dir /content/drive/MyDrive/dynaicl_stage05/runs 2>&1 | tee -a /content/drive/MyDrive/dynaicl_stage05/log.txt

## 5. Summary and archive (the run directory is already on Drive; this also downloads a tar.gz)

In [ ]:
import glob, os, tarfile
run = sorted(glob.glob("/content/drive/MyDrive/dynaicl_stage05/runs/*_stage05_population_advantage_*"))[-1]
print(run)
print(open(os.path.join(run, "results", "report.md")).read()[:6000])
archive = "/content/" + os.path.basename(run) + ".tar.gz"
with tarfile.open(archive, "w:gz") as tf:
    tf.add(run, arcname=os.path.basename(run))
os.system(f"cp {archive} /content/drive/MyDrive/dynaicl_stage05/")
from google.colab import files
files.download(archive)